# Phase 1 — Ground Truth Data Sanity Check

This notebook evaluates the synthetic training (`data/train.csv`) and evaluation (`data/eval.csv`) datasets generated by `src/simulation/generator.py` to confirm:
1. **Hard declines** (`stolen_card`, `do_not_honor`) strictly maintain $P(\text{success}) \approx 0.02 - 0.05$ across all actions.
2. **Soft declines** (`insufficient_funds`, `issuer_unavailable`, `expired_card`) show realistic variation by action, payday proximity, and retry count.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load datasets
train_df = pd.read_csv("../data/train.csv")
eval_df = pd.read_csv("../data/eval.csv")

print(f"Train dataset shape: {train_df.shape}")
print(f"Eval dataset shape: {eval_df.shape}")
train_df.head()

## 1. Decline Code Distribution & True P(Success) Summary

In [ ]:
print("=== True P(Success) Summary by Decline Code ===")
decline_stats = train_df.groupby("decline_code")["true_p_success"].agg(["count", "mean", "std", "min", "max"])
print(decline_stats)

plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="decline_code", y="true_p_success", palette="Set2")
plt.title("Ground-Truth P(Success) Distribution by Decline Code")
plt.ylabel("True P(Success)")
plt.ylim(0, 1.0)
plt.tight_layout()
plt.savefig("../data/decline_code_p_success.png", dpi=150)
plt.show()

## 2. Action Effect Across Decline Codes

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=train_df,
    x="decline_code",
    y="true_p_success",
    hue="action",
    ci=None,
    palette="viridis"
)
plt.title("Mean True P(Success) by Decline Code and Action")
plt.ylabel("Mean True P(Success)")
plt.ylim(0, 1.0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("../data/action_decline_interaction.png", dpi=150)
plt.show()

## 3. Payday Proximity Effect on Insufficient Funds

In [ ]:
inf_df = train_df[train_df["decline_code"] == "insufficient_funds"].copy()
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=inf_df,
    x="day_of_month",
    y="true_p_success",
    hue="action",
    alpha=0.7,
    palette="tab10"
)
plt.title("Insufficient Funds: Day of Month vs True P(Success)")
plt.xlabel("Day of Month (1-31)")
plt.ylabel("True P(Success)")
plt.axvline(x=1, color='red', linestyle='--', alpha=0.5, label="Payday (1st)")
plt.axvline(x=30, color='red', linestyle='--', alpha=0.5, label="Payday (30th/31st)")
plt.tight_layout()
plt.savefig("../data/payday_effect.png", dpi=150)
plt.show()